# Phân Tích 6 Khía Cạnh Hiệu Suất Khoản Vay

Notebook này thực hiện phân tích chi tiết 6 khía cạnh của hiệu suất khoản vay thế chấp:

1. **Delinquency** - Tỷ lệ nợ quá hạn
2. **Default** - Tỷ lệ vỡ nợ
3. **Prepayment** - Tỷ lệ trả trước hạn
4. **Loss Severity** - Mức độ tổn thất
5. **Current UPB** - Số dư gốc hiện tại
6. **Credit Performance Score** - Điểm hiệu suất tín dụng

## 1. Cài Đặt và Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Import thành công!")

## 2. Định Nghĩa Cấu Trúc Dữ Liệu

In [ ]:
# Cấu trúc cột cho performance data (35 cột)
PERF_COLUMNS = [
    'loan_id', 'monthly_reporting_period', 'current_upb', 
    'current_loan_delinquency_status', 'loan_age', 'remaining_months_to_legal_maturity',
    'repurchase_flag', 'modification_flag', 'zero_balance_code', 
    'zero_balance_effective_date', 'current_interest_rate',
    'current_deferred_upb', 'due_date_of_last_paid_installment',
    'mi_recoveries', 'net_sales_proceeds', 'non_mi_recoveries',
    'expenses', 'legal_costs', 'maintenance_and_preservation_costs',
    'taxes_and_insurance', 'miscellaneous_expenses', 'actual_loss_calculation',
    'modification_cost', 'step_modification_flag', 'deferred_payment_plan',
    'estimated_loan_to_value', 'zero_balance_removal_upb', 'delinquent_accrued_interest',
    'delinquency_due_to_disaster', 'borrower_assistance_status_code',
    'current_month_modification_cost', 'interest_bearing_upb',
    'payment_deferral_modification', 'last_paid_installment_date', 'filler'
]

# Cấu trúc cột cho origination data (31 cột)
ORIG_COLUMNS = [
    'credit_score', 'first_payment_date', 'first_time_homebuyer_flag',
    'maturity_date', 'msa', 'mi_percentage', 'number_of_units',
    'occupancy_status', 'cltv', 'dti', 'original_upb', 'ltv',
    'original_interest_rate', 'channel', 'prepayment_penalty_flag',
    'product_type', 'property_state', 'property_type', 'postal_code',
    'loan_id', 'loan_purpose', 'original_loan_term', 'number_of_borrowers',
    'seller_name', 'servicer_name', 'super_conforming_flag',
    'pre_harp_loan_id', 'program_indicator', 'harp_indicator',
    'valuation_method', 'interest_only_indicator', 'filler'
]

print(f"Performance columns: {len(PERF_COLUMNS)}")
print(f"Origination columns: {len(ORIG_COLUMNS)}")

## 3. Load Dữ Liệu

In [ ]:
# Chọn quý để phân tích
QUARTER = '2025Q1'
SAMPLE_SIZE = 50000  # Tăng/giảm tùy theo khả năng máy tính

# Load performance data
perf_file = f'../data/historical_data_2025/perf_{QUARTER}.txt'
print(f"Đang load {perf_file}...")
perf_df = pd.read_csv(
    perf_file, 
    sep='|', 
    names=PERF_COLUMNS,
    nrows=SAMPLE_SIZE,
    dtype={'loan_id': str, 'monthly_reporting_period': str}
)

# Load origination data
orig_file = f'../data/historical_data_2025/orig_{QUARTER}.txt'
print(f"Đang load {orig_file}...")
orig_df = pd.read_csv(
    orig_file,
    sep='|',
    names=ORIG_COLUMNS,
    dtype={'loan_id': str}
)

print(f"\n✓ Load thành công!")
print(f"  Performance: {len(perf_df):,} records")
print(f"  Origination: {len(orig_df):,} records")

## 4. Khám Phá Dữ Liệu

In [ ]:
# Xem cấu trúc performance data
print("=" * 80)
print("PERFORMANCE DATA")
print("=" * 80)
print(perf_df.head())
print(f"\nShape: {perf_df.shape}")
print(f"\nInfo:")
print(perf_df.info())

In [ ]:
# Xem cấu trúc origination data
print("=" * 80)
print("ORIGINATION DATA")
print("=" * 80)
print(orig_df.head())
print(f"\nShape: {orig_df.shape}")
print(f"\nInfo:")
print(orig_df.info())

## 5. Tính Toán 6 Khía Cạnh

In [ ]:
# Merge data
df = perf_df.merge(
    orig_df[['loan_id', 'credit_score', 'original_upb', 'ltv', 'dti']], 
    on='loan_id', 
    how='left'
)

print(f"Merged data: {len(df):,} records")
print(f"Unique loans: {df['loan_id'].nunique():,}")

### 5.1. Dimension 1: Delinquency (Nợ quá hạn)

In [ ]:
# Tính delinquency status
df['delinquency_status'] = pd.to_numeric(
    df['current_loan_delinquency_status'], errors='coerce'
).fillna(0)

df['is_delinquent'] = (df['delinquency_status'] > 0).astype(int)
df['is_seriously_delinquent'] = (df['delinquency_status'] >= 3).astype(int)

# Thống kê
print("DIMENSION 1: DELINQUENCY")
print("=" * 60)
print(f"Tỷ lệ nợ quá hạn: {df['is_delinquent'].mean() * 100:.3f}%")
print(f"Tỷ lệ nợ quá hạn nghiêm trọng (90+ days): {df['is_seriously_delinquent'].mean() * 100:.3f}%")
print(f"\nPhân bố delinquency status:")
print(df['delinquency_status'].value_counts().sort_index())

### 5.2. Dimension 2: Default (Vỡ nợ)

In [ ]:
# Tính default status
default_codes = ['03', '06', '09']
df['is_default'] = df['zero_balance_code'].isin(default_codes).astype(int)

print("DIMENSION 2: DEFAULT")
print("=" * 60)
print(f"Tỷ lệ vỡ nợ: {df['is_default'].mean() * 100:.3f}%")
print(f"\nPhân bố zero_balance_code:")
print(df['zero_balance_code'].value_counts())

### 5.3. Dimension 3: Prepayment (Trả trước hạn)

In [ ]:
# Tính prepayment status
df['is_prepaid'] = (df['zero_balance_code'] == '01').astype(int)

print("DIMENSION 3: PREPAYMENT")
print("=" * 60)
print(f"Tỷ lệ trả trước hạn: {df['is_prepaid'].mean() * 100:.3f}%")

### 5.4. Dimension 4: Loss Severity (Mức độ tổn thất)

In [ ]:
# Tính loss severity
df['loss_severity'] = (
    pd.to_numeric(df['actual_loss_calculation'], errors='coerce').fillna(0) / 
    df['original_upb'].replace(0, np.nan)
).fillna(0)

print("DIMENSION 4: LOSS SEVERITY")
print("=" * 60)
print(f"Mức độ tổn thất trung bình: {df['loss_severity'].mean() * 100:.3f}%")
print(f"Mức độ tổn thất median: {df['loss_severity'].median() * 100:.3f}%")
print(f"Mức độ tổn thất max: {df['loss_severity'].max() * 100:.3f}%")

### 5.5. Dimension 5: Current UPB (Số dư gốc)

In [ ]:
# Tính current UPB
df['current_upb'] = pd.to_numeric(df['current_upb'], errors='coerce').fillna(0)

print("DIMENSION 5: CURRENT UPB")
print("=" * 60)
print(f"Tổng số dư gốc: ${df['current_upb'].sum():,.2f}")
print(f"Số dư gốc trung bình: ${df['current_upb'].mean():,.2f}")
print(f"Số dư gốc median: ${df['current_upb'].median():,.2f}")

### 5.6. Dimension 6: Credit Performance Score

In [ ]:
# Tính credit performance score
def calculate_performance_score(row):
    score = 100.0
    score -= row['delinquency_status'] * 10
    score -= row['is_default'] * 50
    credit_score_normalized = (row['credit_score'] if pd.notna(row['credit_score']) else 700 - 300) / 550 * 20
    score += credit_score_normalized
    return max(0, min(100, score))

df['credit_performance_score'] = df.apply(calculate_performance_score, axis=1)

print("DIMENSION 6: CREDIT PERFORMANCE SCORE")
print("=" * 60)
print(f"Điểm hiệu suất trung bình: {df['credit_performance_score'].mean():.2f}")
print(f"Điểm hiệu suất median: {df['credit_performance_score'].median():.2f}")
print(f"Điểm hiệu suất min-max: {df['credit_performance_score'].min():.2f} - {df['credit_performance_score'].max():.2f}")

## 6. Tổng Hợp Theo Thời Gian

In [ ]:
# Tổng hợp theo tháng
monthly_agg = df.groupby('monthly_reporting_period').agg({
    'is_delinquent': 'mean',
    'is_seriously_delinquent': 'mean',
    'is_default': 'mean',
    'is_prepaid': 'mean',
    'loss_severity': 'mean',
    'current_upb': 'sum',
    'credit_performance_score': 'mean',
    'loan_id': 'count'
})

monthly_agg.columns = [
    'delinquency_rate', 'serious_delinquency_rate',
    'default_rate', 'prepayment_rate', 'avg_loss_severity',
    'total_upb', 'avg_credit_score', 'loan_count'
]

# Chuyển sang phần trăm
rate_cols = ['delinquency_rate', 'serious_delinquency_rate', 'default_rate', 'prepayment_rate']
monthly_agg[rate_cols] = monthly_agg[rate_cols] * 100

print("\nTỔNG HỢP THEO THÁNG")
print("=" * 80)
print(monthly_agg)

## 7. Visualization

In [ ]:
# Vẽ biểu đồ 6 dimensions
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle(f'6 Khía Cạnh Hiệu Suất Khoản Vay - {QUARTER}', fontsize=16, fontweight='bold')

monthly_agg_plot = monthly_agg.copy()
monthly_agg_plot.index = pd.to_datetime(monthly_agg_plot.index, format='%Y%m')

# 1. Delinquency
ax = axes[0, 0]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['delinquency_rate'], marker='o', label='Tổng', linewidth=2)
ax.plot(monthly_agg_plot.index, monthly_agg_plot['serious_delinquency_rate'], marker='s', label='Nghiêm trọng', linewidth=2)
ax.set_title('1. Tỷ Lệ Nợ Quá Hạn (%)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Default
ax = axes[0, 1]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['default_rate'], marker='o', color='red', linewidth=2)
ax.set_title('2. Tỷ Lệ Vỡ Nợ (%)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 3. Prepayment
ax = axes[1, 0]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['prepayment_rate'], marker='o', color='green', linewidth=2)
ax.set_title('3. Tỷ Lệ Trả Trước (%)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 4. Loss Severity
ax = axes[1, 1]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['avg_loss_severity'] * 100, marker='o', color='orange', linewidth=2)
ax.set_title('4. Mức Độ Tổn Thất (%)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 5. Total UPB
ax = axes[2, 0]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['total_upb'] / 1e9, marker='o', color='purple', linewidth=2)
ax.set_title('5. Tổng Số Dư (Billion $)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 6. Credit Score
ax = axes[2, 1]
ax.plot(monthly_agg_plot.index, monthly_agg_plot['avg_credit_score'], marker='o', color='blue', linewidth=2)
ax.set_title('6. Điểm Hiệu Suất', fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Phân Tích Chi Tiết

### 8.1. Phân tích theo Credit Score

In [ ]:
# Phân nhóm theo credit score
df['credit_score_bin'] = pd.cut(
    df['credit_score'], 
    bins=[0, 620, 680, 740, 800, 900],
    labels=['<620 (Poor)', '620-680 (Fair)', '680-740 (Good)', '740-800 (Very Good)', '800+ (Excellent)']
)

score_analysis = df.groupby('credit_score_bin').agg({
    'is_delinquent': 'mean',
    'is_default': 'mean',
    'is_prepaid': 'mean',
    'current_upb': 'mean',
    'loan_id': 'count'
})

score_analysis.columns = ['Delinquency Rate', 'Default Rate', 'Prepayment Rate', 'Avg UPB', 'Count']
score_analysis[['Delinquency Rate', 'Default Rate', 'Prepayment Rate']] *= 100

print("\nPHÂN TÍCH THEO CREDIT SCORE")
print("=" * 80)
print(score_analysis)

### 8.2. Phân tích theo Loan Age

In [ ]:
# Phân nhóm theo loan age
df['loan_age_bin'] = pd.cut(
    pd.to_numeric(df['loan_age'], errors='coerce'),
    bins=[0, 12, 24, 36, 60, 1000],
    labels=['0-12 months', '12-24 months', '24-36 months', '36-60 months', '60+ months']
)

age_analysis = df.groupby('loan_age_bin').agg({
    'is_delinquent': 'mean',
    'is_default': 'mean',
    'is_prepaid': 'mean',
    'credit_performance_score': 'mean',
    'loan_id': 'count'
})

age_analysis.columns = ['Delinquency Rate', 'Default Rate', 'Prepayment Rate', 'Avg Score', 'Count']
age_analysis[['Delinquency Rate', 'Default Rate', 'Prepayment Rate']] *= 100

print("\nPHÂN TÍCH THEO LOAN AGE")
print("=" * 80)
print(age_analysis)

### 8.3. Correlation Matrix

In [ ]:
# Tính correlation
corr_cols = [
    'delinquency_status', 'is_default', 'is_prepaid', 
    'loss_severity', 'credit_performance_score',
    'credit_score', 'ltv', 'dti'
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Kết Luận

### Các phát hiện chính:

1. **Delinquency**: [Điền kết quả phân tích]
2. **Default**: [Điền kết quả phân tích]
3. **Prepayment**: [Điền kết quả phân tích]
4. **Loss Severity**: [Điền kết quả phân tích]
5. **UPB**: [Điền kết quả phân tích]
6. **Credit Score**: [Điền kết quả phân tích]

### Khuyến nghị:

1. [Khuyến nghị 1]
2. [Khuyến nghị 2]
3. [Khuyến nghị 3]